# SituatiONION V5: Decodability Atlas

This notebook maps held-out linear decodability across situation variables, token readouts, and GPT-2 XL layers. It uses V4's frozen long-format data and hidden-state cache. Decodability is not causal necessity.

# V5- Initial idea.

**Central question: What specific components of the situation can be decoded at different depths?**

V5 would contain:

Layer-wise probes.
Agent decoding.
Recipient decoding.
Event decoding.
Causality decoding.
Temporal-relation decoding.
Polarity decoding.
Held-out-template evaluation.
Attention comparison.
PCA vs random-projection controls.
Stronger non-semantic controls if V4 identifies remaining concerns.

This is where you investigate whether the geometry from V4 actually corresponds to identifiable situation variables.

Your progression becomes:

$$ \text{V4: Where is information?} $$ $$ \downarrow $$ $$ \text{V5: What information is there?} $$

## Held-Out Linear Probes

Identity probes test only held-out templates with seen entities. Binary situation-variable probes test templates, entities, and combined holdouts. Each task reports linear, shuffled-label, random-projection, lexical, and majority controls.

# V5 — Decodability and Mechanistic Diagnostics

## Central Question

**Which situation variables are decodable from which token representations at different depths, and how does their decodability change across the network?**

V4 established that situation-sensitive geometry is distributed across GPT-2 XL rather than localized to a single privileged layer. It also showed that this geometry depends strongly on both the **situation dimension being manipulated** and the **token representation being measured**.

V5 therefore moves from measuring geometric separation to identifying the information represented by that geometry.

$$
\text{V4: Where is situation-sensitive information?}
$$

$$
\downarrow
$$

$$
\text{V5: What situation information is represented there?}
$$

## Primary Analyses

### 1. Layer × Readout Probing

Train layer-wise probes at multiple token representations rather than probing only one sentence-level representation.

Readouts should include:

* Changed token
* Event token
* Final token
* Agent/recipient token where applicable
* Mean pool as a global baseline
* Max pool as a negative/weak baseline where useful

The goal is to construct a **layer × readout × situation-variable** decodability map.

### 2. Situation-Variable Decoding

Probe separately for:

* Agent
* Recipient
* Agent/recipient role assignment
* Event/state
* Causality
* Polarity
* Temporal relation

These variables should be treated as separate prediction problems rather than collapsed into a single "situation" label.

### 3. Held-Out Generalization

Probe performance must be evaluated on held-out templates or structural families.

Random train/test sentence splits are insufficient because a probe could exploit lexical or template-specific regularities rather than situation information.

Where possible, evaluate:

* Held-out templates
* Held-out lexical items
* Held-out situation combinations

This asks whether the representation encodes a generalizable situation variable rather than memorizing the construction used to express it.

### 4. Probe Controls

For every probe, compare against appropriate controls:

* Majority/chance baseline
* Label-shuffled probe
* Lexical/surface-feature baseline
* Random-projection representation control
* Where appropriate, embedding-layer or early-layer baseline

Probe complexity should be kept deliberately limited, beginning with linear probes.

The purpose is to measure information that is readily available in the representation, not information that a powerful classifier can reconstruct.

### 5. V4-Guided Predictions

V5 should preregister several predictions based on V4.

**P1 — Token specificity**

Situation variables should generally be more decodable from semantically relevant token representations than from global pooled representations.

**P2 — Changed-token early availability**

Variables directly expressed by the manipulated token may already be decodable in very early layers.

This would distinguish **local lexical availability** from later contextual integration.

**P3 — Event-token contextual development**

Event/state information should show increasing decodability at the Event-token representation through early and middle layers, consistent with the depth-dependent trajectory observed in V4.

**P4 — Final-token integration**

Multiple situation variables may become decodable from the Final-token representation, consistent with contextual information becoming distributed beyond its original token location.

**P5 — Dimension-specific trajectories**

Agent/recipient, event/state, causality, polarity, and temporal relations should not exhibit identical layer-wise decodability curves.

**P6 — Temporal uncertainty**

Because V4 found weak temporal separation, temporal decoding is an important diagnostic rather than an expected positive result.

Failure to decode temporal relations would itself constrain the broader situation-model hypothesis.

### 6. Geometry–Decodability Correspondence

Compare V4 geometric separation with V5 probe performance.

For each:

$$
(\text{dimension},\text{readout},\text{layer})
$$

ask whether stronger counterfactual-vs-paraphrase separation predicts greater decodability of the corresponding situation variable.

This directly tests whether the geometry identified in V4 corresponds to identifiable semantic information.

## Secondary Mechanistic Diagnostics

### Attention Analysis

Compare attention patterns only as a secondary diagnostic.

Attention should not by itself be interpreted as evidence that a component stores or causally uses situation information.

Ask whether changes in decodability coincide with changes in information flow between situation-relevant token positions.

### PCA vs Random Projection

Retain the random-projection comparison from V4 as a visualization control.

PCA should be treated as descriptive visualization rather than primary evidence.

The primary quantitative evidence in V5 should come from held-out decoding performance and its controls.

## Interpretation Boundary

V5 tests **decodability**, not causal necessity.

Successful decoding establishes that information about a situation variable is available in a representation under the probe's assumptions. It does not establish that GPT-2 XL itself uses that information to produce its predictions.

That distinction is reserved for V6:

$$
\text{V4: Where is information?}
$$

$$
\downarrow
$$

$$
\text{V5: What information is there?}
$$

$$
\downarrow
$$

$$
\text{V6: Does the model causally use it?}
$$

## V5 Goal

Produce a controlled map of:

$$
\boxed{
\text{Situation Variable}
\times
\text{Token Readout}
\times
\text{Layer}
\rightarrow
\text{Decodability}
}
$$

The central V5 result should therefore not be a single "best layer."

It should be a **decodability atlas showing what information is available where, and how that availability changes through the network.**


# 1. Setup and inputs
   Load data/v5_probe_examples.csv, define output paths, seeds, all 48 layers, the target variables, and the six token readouts. Load the cached V4 GPT-2 XL hidden states from:
results/layer_curves/gpt2xl_hidden_states.pt


In [4]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: mount failed

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/fall26/coding/SituationionArtifacts"
)
DATA_PATH = ARTIFACT_DIR / "v5_probe_examples.csv"
CACHE_PATH = ARTIFACT_DIR / "gpt2xl_hidden_states.pt"

In [ ]:
!ls -lh /content/drive/MyDrive/fall26/coding/SituationionArtifacts


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.random_projection import GaussianRandomProjection

ROOT = Path.cwd()

In [ ]:
ROOT = Path.cwd()

In [ ]:
ROOT = Path.cwd()
# DATA_PATH = ROOT / 'data' / 'v5_probe_examples.csv'
# CACHE_PATH = ROOT / 'results' / 'layer_curves' / 'gpt2xl_hidden_states.pt'

OUTPUT_DIR = ARTIFACT_DIR / "v5_decodability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED, LAYERS = 7, range(48)
rng = np.random.default_rng(SEED)
TASKS = {'agent_identity': ('semantic_agent', 'agent_identity_eligible'), 'recipient_identity': ('semantic_recipient', 'recipient_identity_eligible'), 'role_assignment': ('role_assignment', 'role_assignment_eligible'), 'event_state': ('event_state', 'event_state_eligible'), 'cause_holder_role': ('cause_holder_role', 'cause_holder_role_eligible'), 'polarity': ('polarity', 'polarity_eligible'), 'temporal_relation': ('temporal_relation', 'temporal_relation_eligible')}
READOUTS = ('changed_token', 'event_token', 'final_token', 'agent_recipient', 'mean_pool', 'max_pool')
if not DATA_PATH.exists(): raise FileNotFoundError('Run python3 structural_controls.py first.')
if not CACHE_PATH.exists(): raise FileNotFoundError('Copy gpt2xl_hidden_states.pt from V4 into results/layer_curves/.')
probe_examples = pd.read_csv(DATA_PATH)
cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
tokenizer, RUNS = cache['tokenizer'], cache['runs']
print(f'Loaded {len(probe_examples)} sentences and {len(RUNS)} cached triples.')


# 2. Readout extraction
   Write functions that return a representation for one sentence at one layer:
   - Changed-token hidden state
   - Event-token hidden state
   - Final-token hidden state
   - Ordered agent/recipient representation
   - Mean pool
   - Max pool
The long-format CSV supplies triple_id, variant, semantic role labels, and eligibility flags; the hidden-state cache supplies the actual tensors.


In [ ]:
EVENT = {'agent_recipient': ('gave', 'gave', 'gave'), 'cause': ('called', 'called', 'called'), 'temporal': ('cleaned', 'cleaned', 'cleaned'), 'polarity': ('repair', 'mend', 'fix'), 'event_state': ('carried', 'transported', 'dropped')}
CHANGED = {'temporal': ('after', 'once', 'before'), 'polarity': ('repair', 'mend', 'not'), 'event_state': ('carried', 'transported', 'dropped')}
def token_position(text, surface):
    matches = list(re.finditer(r'(?<![a-zA-Z])' + re.escape(surface) + r'(?![a-zA-Z])', text, re.I))
    if not matches: raise ValueError(f'{surface!r} was not found in {text!r}')
    return len(tokenizer.encode(text[:matches[-1].end()], add_special_tokens=False)) - 1
def readout_vector(row, layer, readout):
    run = RUNS[row.triple_id][row.variant]; hidden = run['hidden_states'][layer + 1]
    index = ('base', 'paraphrase', 'counterfactual').index(row.variant)
    if readout == 'mean_pool': return hidden.mean(0).numpy()
    if readout == 'max_pool': return hidden.max(0).values.numpy()
    if readout == 'final_token': return hidden[-1].numpy()
    if readout == 'event_token': return hidden[token_position(run['text'], EVENT[row.manipulation][index])].numpy()
    if readout == 'changed_token':
        surface = row.semantic_recipient if row.manipulation == 'agent_recipient' and row.variant == 'counterfactual' else (row.semantic_agent if row.manipulation in {'agent_recipient', 'cause'} else CHANGED[row.manipulation][index])
        return hidden[token_position(run['text'], surface)].numpy()
    return torch.cat((hidden[token_position(run['text'], row.semantic_agent)], hidden[token_position(run['text'], row.semantic_recipient)])).numpy()
test_vector = readout_vector(probe_examples.iloc[0], 0, 'final_token')
print('Readout extraction ready:', test_vector.shape)


# 3. Held-out linear probes
   For each valid combination of:
task × readout × layer × evaluation split
fit a class-balanced logistic-regression probe on split == "dev" and evaluate separately on:
- test_template
- test_entity
- test_both
Use balanced accuracy. Identity tasks should use only held-out templates with known entity labels. Do not run identity classification on unseen entity names.


In [ ]:
def probe_score(x_train, y_train, x_test, y_test):
    if len(set(y_train)) < 2 or len(set(y_test)) < 2: return np.nan
    probe = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
    return balanced_accuracy_score(y_test, probe.fit(x_train, y_train).predict(x_test))
def eligible_splits(task, data):
    return ('test_template',) if task in {'agent_identity', 'recipient_identity'} else tuple(split for split in ('test_template', 'test_entity', 'test_both') if (data.split == split).any())
linear_rows = []
for task, (label, flag) in TASKS.items():
    data = probe_examples[probe_examples[flag]].copy(); train = data.query("split == 'dev'")
    for test_split in eligible_splits(task, data):
        test = data.query('split == @test_split')
        for readout in READOUTS:
            for layer in LAYERS:
                x_train = np.stack([readout_vector(row, layer, readout) for row in train.itertuples(index=False)])
                x_test = np.stack([readout_vector(row, layer, readout) for row in test.itertuples(index=False)])
                linear_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'linear', 'score': probe_score(x_train, train[label].to_numpy(), x_test, test[label].to_numpy())})
linear_atlas = pd.DataFrame(linear_rows)
print(f'Finished {len(linear_atlas)} held-out linear probes.')


# 4. Controls
   For every probe result, include:
   - Majority-class baseline
   - Label-shuffled logistic probe
   - TF-IDF word/bigram logistic probe
   - Random-projected hidden-state probe
   - Layer 0 as the early/embedding baseline
Store all scores in one CSV, for example:
results/v5_decodability/decodability_atlas.csv
with columns such as task, readout, layer, test_split, control, and score.


In [ ]:
control_rows = []
for task, (label, flag) in TASKS.items():
    data = probe_examples[probe_examples[flag]].copy(); train = data.query("split == 'dev'")
    for test_split in eligible_splits(task, data):
        test = data.query('split == @test_split'); y_train, y_test = train[label].to_numpy(), test[label].to_numpy()
        majority = train[label].value_counts(normalize=True).max()
        control_rows.append({'task': task, 'readout': 'majority', 'layer': -1, 'test_split': test_split, 'control': 'majority', 'score': majority})
        tfidf = TfidfVectorizer(ngram_range=(1, 2)); x_train = tfidf.fit_transform(train.text); x_test = tfidf.transform(test.text)
        control_rows.append({'task': task, 'readout': 'lexical_tfidf', 'layer': -1, 'test_split': test_split, 'control': 'lexical', 'score': probe_score(x_train, y_train, x_test, y_test)})
        for readout in READOUTS:
            for layer in LAYERS:
                x_train = np.stack([readout_vector(row, layer, readout) for row in train.itertuples(index=False)]); x_test = np.stack([readout_vector(row, layer, readout) for row in test.itertuples(index=False)])
                control_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'shuffled_labels', 'score': probe_score(x_train, rng.permutation(y_train), x_test, y_test)})
                rp = GaussianRandomProjection(n_components=min(128, x_train.shape[1]), random_state=SEED)
                control_rows.append({'task': task, 'readout': readout, 'layer': layer, 'test_split': test_split, 'control': 'random_projection', 'score': probe_score(rp.fit_transform(x_train), y_train, rp.transform(x_test), y_test)})
atlas = pd.concat([linear_atlas, pd.DataFrame(control_rows)], ignore_index=True)
atlas.to_csv(OUTPUT_DIR / 'decodability_atlas.csv', index=False)
display(atlas.groupby('control').size().rename('n_results'))


# 5. Results and interpretation
   Create heatmaps with:
   - Rows: readouts
   - Columns: layers
   - Separate panels: situation variables
   - Values: held-out balanced accuracy
Save the atlas figures and a table of each task/readout’s peak score and layer. If results/layer_curves/heldout_layer_curves.csv is available from V4, join it by readout and layer and calculate correlations between V4 separation and V5 decodability.

In [ ]:
for test_split in ('test_template', 'test_entity', 'test_both'):
    subset = atlas.query("test_split == @test_split and control == 'linear'")
    if subset.empty: continue
    tasks = subset.task.unique(); fig, axes = plt.subplots(len(tasks), 1, figsize=(15, 2.2 * len(tasks)), sharex=True); axes = np.atleast_1d(axes)
    for axis, task in zip(axes, tasks):
        matrix = subset.query('task == @task').pivot(index='readout', columns='layer', values='score').reindex(READOUTS)
        image = axis.imshow(matrix, aspect='auto', origin='lower', vmin=0, vmax=1, cmap='viridis')
        axis.set(yticks=range(len(READOUTS)), yticklabels=READOUTS, ylabel=task); axis.axvline(23, color='white'); axis.axvline(26, color='white')
    axes[-1].set_xlabel('GPT-2 XL block'); fig.colorbar(image, ax=axes, label='Held-out balanced accuracy'); fig.suptitle(f'V5 atlas: {test_split}', y=1.01); fig.tight_layout(); fig.savefig(OUTPUT_DIR / f'atlas_{test_split}.png', dpi=180, bbox_inches='tight'); plt.show()
summary = atlas.query("control == 'linear'").groupby(['task', 'readout', 'test_split']).score.agg(['max', 'idxmax']).reset_index(); summary.to_csv(OUTPUT_DIR / 'decodability_summary.csv', index=False); display(summary)
v4_path = ROOT / 'results' / 'layer_curves' / 'heldout_layer_curves.csv'
if v4_path.exists():
    geometry = pd.read_csv(v4_path).rename(columns={'mean': 'geometry'}); joined = atlas.query("control == 'linear'").merge(geometry[['readout', 'layer', 'geometry']], on=['readout', 'layer'])
    correlations = joined.groupby(['task', 'readout', 'test_split']).apply(lambda frame: frame.geometry.corr(frame.score)).rename('pearson_r').reset_index(); correlations.to_csv(OUTPUT_DIR / 'geometry_decodability_correlations.csv', index=False); display(correlations.round(3))
else: print('V4 numeric curves are absent; atlas saved, correspondence skipped.')
print('Interpretation boundary: V5 measures decodability; V6 tests causal necessity.')
